# Outlier diagnostics notebook

Aquest notebook serveix per:

- carregar el `*_pitch_preprocessed_debug.parquet`
- reconstruir **per què** s'ha marcat cada `is_outlier`
- separar el criteri de **velocitat** del criteri de **desviació respecte a mediana local**
- comparar els **outliers guardats al dataframe** amb uns **outliers recalculats** amb nous paràmetres
- visualitzar i escoltar una finestra temporal amb **dos subplots**:
  - dalt: outliers antics / guardats
  - baix: outliers nous / tuned

La columna sobre la qual es calculen els outliers és, per defecte, **`f0_interpolated`**.

In [1]:

from pathlib import Path
import numpy as np
import polars as pl
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
import soundfile as sf
import sys

from IPython.display import display, Audio, Javascript

try:
    from settings import PROJECT_ROOT
except Exception:
    PROJECT_ROOT = Path(".").resolve()

def find_project_root(start=None):
    if start is None:
        start = Path.cwd().resolve()

    for p in [start] + list(start.parents):
        if (p / "data").exists():
            return p

    return start


PROJECT_ROOT = find_project_root()
sys.path.append(str(PROJECT_ROOT))

plt.rcParams["figure.figsize"] = (14, 5)

In [ ]:
import settings as S


RECORDING_ID = S.SARASUDA_VARNAM[0]
TONIC_HZ = S.SARASUDA_TONICS[RECORDING_ID]
ROOT_DIR = "data/interim"

# si el detectem automàticament, no cal tocar-ho
AUDIO_PATH = None

# paràmetres "antics" del pipeline
OLD_MAX_VELOCITY_STS = 200.0
OLD_DEV_THRESH_CENTS = 600.0
OLD_EXPAND_NEIGHBORS = 1

# paràmetres "nous" inicials per comparar
NEW_MAX_VELOCITY_STS = 300.0
NEW_DEV_THRESH_CENTS = 600.0
NEW_EXPAND_NEIGHBORS = 1

In [3]:

def resolve_path(p):
    p = Path(p)
    if p.is_absolute():
        return p
    return Path(PROJECT_ROOT) / p

def get_debug_pitch_path(recording_id, root_dir="data/interim"):
    root_dir = resolve_path(root_dir)
    return root_dir / recording_id / "pitch" / f"{recording_id}_pitch_preprocessed_debug.parquet"

def find_audio_path(recording_id):
    corpus_root = resolve_path("data/corpus") / recording_id / "audio"
    if not corpus_root.exists():
        return None
    exts = [".wav", ".mp3", ".flac", ".ogg", ".m4a"]
    files = []
    for ext in exts:
        files.extend(sorted(corpus_root.glob(f"*{ext}")))
    return files[0] if files else None

debug_path = get_debug_pitch_path(RECORDING_ID, ROOT_DIR)
df_debug = pl.read_parquet(debug_path)

if AUDIO_PATH is None:
    AUDIO_PATH = find_audio_path(RECORDING_ID)

print("debug_path:", debug_path)
print("audio_path:", AUDIO_PATH)
print("shape:", df_debug.shape)
display(df_debug.head().to_pandas())

debug_path: /home/lluis/master-thesis/CSISD/data/interim/srs_v1_bdn_sav/pitch/srs_v1_bdn_sav_pitch_preprocessed_debug.parquet
audio_path: /home/lluis/master-thesis/CSISD/data/corpus/srs_v1_bdn_sav/audio/srs_v1_bdn_sav.mp3
shape: (43805, 10)


,time_rel_sec,f0_Hz,f0_pchip,f0_savgol_p3_w13,f0_savgol_p3_w27,f0_interpolated,too_long_to_interp,is_outlier,valid_for_pchip,group_id
0,0.000000,0.0,NaN,NaN,NaN,0.0,True,False,False,NaN
1,0.010001,0.0,NaN,NaN,NaN,0.0,True,False,False,0.0
2,0.020001,0.0,NaN,NaN,NaN,0.0,True,False,False,0.0
3,0.030002,0.0,NaN,NaN,NaN,0.0,True,False,False,0.0
4,0.040003,0.0,NaN,NaN,NaN,0.0,True,False,False,0.0


In [4]:

def hz_to_cents(f_hz, tonic_hz):
    f_hz = np.asarray(f_hz, dtype=float)
    out = np.full_like(f_hz, np.nan, dtype=float)
    mask = np.isfinite(f_hz) & (f_hz > 0)
    out[mask] = 1200.0 * np.log2(f_hz[mask] / tonic_hz)
    return out

for col in ["f0_Hz", "f0_pchip", "f0_savgol_p3_w13", "f0_savgol_p3_w27", "f0_interpolated"]:
    if col in df_debug.columns and f"{col}_cents" not in df_debug.columns:
        df_debug = df_debug.with_columns(
            pl.Series(f"{col}_cents", hz_to_cents(df_debug[col].to_numpy(), TONIC_HZ))
        )

df_debug.columns

['time_rel_sec',
 'f0_Hz',
 'f0_pchip',
 'f0_savgol_p3_w13',
 'f0_savgol_p3_w27',
 'f0_interpolated',
 'too_long_to_interp',
 'is_outlier',
 'valid_for_pchip',
 'group_id',
 'f0_Hz_cents',
 'f0_pchip_cents',
 'f0_savgol_p3_w13_cents',
 'f0_savgol_p3_w27_cents',
 'f0_interpolated_cents']

## 1. Funcions de diagnòstic dels outliers

La funció següent reprodueix la lògica de `remove_outliers(...)`, però retornant per separat:

- `is_outlier_vel_raw`: criteri de velocitat
- `is_outlier_dev_raw`: criteri de desviació respecte a mediana local
- `is_outlier_raw`: unió dels dos criteris abans d'expandir
- `is_outlier`: màscara final després d'expandir veïns
- `f0_clean`: columna d'entrada amb els outliers posats a `NaN`

In [5]:

from scipy.signal import medfilt

def compute_outlier_diagnostics(
    df: pl.DataFrame,
    col_time: str = "time_rel_sec",
    col_f0: str = "f0_interpolated",
    col_too_long: str = "too_long_to_interp",
    max_velocity_sts: float = 200.0,
    dev_thresh_cents: float = 600.0,
    expand_neighbors: int = 1,
    f_ref=None,
):
    t = df[col_time].to_numpy()
    f = df[col_f0].to_numpy().astype(float)
    too_long = df[col_too_long].fill_null(False).to_numpy().astype(bool)

    N = len(f)
    mask_valid = (f > 0) & (~too_long)
    valid_idx = np.where(mask_valid)[0]

    out = {
        "mask_valid": mask_valid,
        "valid_idx": valid_idx,
        "f_ref": np.nan,
        "f_cents": np.full(N, np.nan, dtype=float),
        "median_local": np.full(N, np.nan, dtype=float),
        "dev_cents": np.full(N, np.nan, dtype=float),
        "vel_jump_cents": np.full(N, np.nan, dtype=float),
        "vel_max_allowed_cents": np.full(N, np.nan, dtype=float),
        "is_outlier_vel_raw": np.zeros(N, dtype=bool),
        "is_outlier_dev_raw": np.zeros(N, dtype=bool),
        "is_outlier_raw": np.zeros(N, dtype=bool),
        "is_outlier": np.zeros(N, dtype=bool),
        "f0_clean": f.copy(),
    }

    if len(valid_idx) < 3:
        return out

    if f_ref is None:
        f_ref = np.nanmedian(f[mask_valid])

    out["f_ref"] = float(f_ref)
    f_cents = out["f_cents"]
    f_cents[mask_valid] = 1200.0 * np.log2(f[mask_valid] / f_ref)

    # --- criteri de velocitat ---
    max_vel_cents = max_velocity_sts * 100.0
    prev_idx = valid_idx[0]
    prev_cents = f_cents[prev_idx]
    prev_time = t[prev_idx]

    for idx in valid_idx[1:]:
        curr_cents = f_cents[idx]
        curr_time = t[idx]
        dt = curr_time - prev_time

        if dt <= 0:
            prev_idx = idx
            prev_cents = curr_cents
            prev_time = curr_time
            continue

        dcents = abs(curr_cents - prev_cents)
        max_d_allowed = max_vel_cents * dt

        out["vel_jump_cents"][idx] = dcents
        out["vel_max_allowed_cents"][idx] = max_d_allowed

        if dcents > max_d_allowed:
            out["is_outlier_vel_raw"][idx] = True

        prev_idx = idx
        prev_cents = curr_cents
        prev_time = curr_time

    # --- criteri de desviació respecte a mediana local ---
    f_cents_valid = f_cents[valid_idx]

    kernel_size = 31
    if kernel_size > len(f_cents_valid):
        kernel_size = len(f_cents_valid) if len(f_cents_valid) % 2 == 1 else len(f_cents_valid) - 1
    if kernel_size < 3:
        kernel_size = 3

    median_local_valid = medfilt(f_cents_valid, kernel_size=kernel_size)
    dev_valid = np.abs(f_cents_valid - median_local_valid)
    is_outlier_dev_local = dev_valid > dev_thresh_cents

    out["median_local"][valid_idx] = median_local_valid
    out["dev_cents"][valid_idx] = dev_valid
    out["is_outlier_dev_raw"][valid_idx] = is_outlier_dev_local

    # unió
    is_outlier = out["is_outlier_vel_raw"] | out["is_outlier_dev_raw"]
    out["is_outlier_raw"] = is_outlier.copy()

    # expansió
    if expand_neighbors > 0:
        is_outlier_expanded = is_outlier.copy()
        for shift in range(1, expand_neighbors + 1):
            is_outlier_expanded[:-shift] |= is_outlier[shift:]
            is_outlier_expanded[shift:] |= is_outlier[:-shift]
        is_outlier = is_outlier_expanded

    is_outlier &= mask_valid
    out["is_outlier"] = is_outlier

    f_clean = f.copy()
    f_clean[is_outlier] = np.nan
    out["f0_clean"] = f_clean

    return out

In [6]:

diag_old = compute_outlier_diagnostics(
    df_debug,
    col_f0="f0_interpolated",
    max_velocity_sts=OLD_MAX_VELOCITY_STS,
    dev_thresh_cents=OLD_DEV_THRESH_CENTS,
    expand_neighbors=OLD_EXPAND_NEIGHBORS,
)

diag_new = compute_outlier_diagnostics(
    df_debug,
    col_f0="f0_interpolated",
    max_velocity_sts=NEW_MAX_VELOCITY_STS,
    dev_thresh_cents=NEW_DEV_THRESH_CENTS,
    expand_neighbors=NEW_EXPAND_NEIGHBORS,
)

df_diag = df_debug.with_columns([
    pl.Series("is_outlier_stored", df_debug["is_outlier"].fill_null(False).to_numpy().astype(bool)),
    pl.Series("is_outlier_old_recomputed", diag_old["is_outlier"]),
    pl.Series("is_outlier_old_vel_raw", diag_old["is_outlier_vel_raw"]),
    pl.Series("is_outlier_old_dev_raw", diag_old["is_outlier_dev_raw"]),
    pl.Series("old_vel_jump_cents", diag_old["vel_jump_cents"]),
    pl.Series("old_vel_max_allowed_cents", diag_old["vel_max_allowed_cents"]),
    pl.Series("old_median_local_cents", diag_old["median_local"]),
    pl.Series("old_dev_cents", diag_old["dev_cents"]),
    pl.Series("is_outlier_new", diag_new["is_outlier"]),
    pl.Series("is_outlier_new_vel_raw", diag_new["is_outlier_vel_raw"]),
    pl.Series("is_outlier_new_dev_raw", diag_new["is_outlier_dev_raw"]),
    pl.Series("new_vel_jump_cents", diag_new["vel_jump_cents"]),
    pl.Series("new_vel_max_allowed_cents", diag_new["vel_max_allowed_cents"]),
    pl.Series("new_median_local_cents", diag_new["median_local"]),
    pl.Series("new_dev_cents", diag_new["dev_cents"]),
])

summary = {
    "stored": int(df_diag["is_outlier_stored"].sum()),
    "old_recomputed": int(df_diag["is_outlier_old_recomputed"].sum()),
    "old_vel_raw": int(df_diag["is_outlier_old_vel_raw"].sum()),
    "old_dev_raw": int(df_diag["is_outlier_old_dev_raw"].sum()),
    "new_total": int(df_diag["is_outlier_new"].sum()),
    "new_vel_raw": int(df_diag["is_outlier_new_vel_raw"].sum()),
    "new_dev_raw": int(df_diag["is_outlier_new_dev_raw"].sum()),
}
summary

{'stored': 1483,
 'old_recomputed': 151,
 'old_vel_raw': 7,
 'old_dev_raw': 102,
 'new_total': 143,
 'new_vel_raw': 0,
 'new_dev_raw': 102}

## 2. Runs d'outliers i taula de diagnòstic

In [7]:

def find_true_runs(mask_bool):
    mask_bool = np.asarray(mask_bool, dtype=bool)
    runs = []
    in_run = False
    start = None

    for i, v in enumerate(mask_bool):
        if v and not in_run:
            start = i
            in_run = True
        elif not v and in_run:
            runs.append((start, i))
            in_run = False

    if in_run:
        runs.append((start, len(mask_bool)))
    return runs

def build_run_table(df: pl.DataFrame, mask_col: str, prefix: str = "old"):
    t = df["time_rel_sec"].to_numpy()
    runs = find_true_runs(df[mask_col].fill_null(False).to_numpy().astype(bool))

    rows = []
    for k, (s, e) in enumerate(runs):
        rows.append({
            "run_id": k,
            "start_idx": s,
            "end_idx_exclusive": e,
            "n_samples": e - s,
            "t_start": float(t[s]),
            "t_end": float(t[e - 1]),
            "t_mid": float(0.5 * (t[s] + t[e - 1])),
            "any_vel_raw": bool(df[f"is_outlier_{prefix}_vel_raw"][s:e].any()) if f"is_outlier_{prefix}_vel_raw" in df.columns else None,
            "any_dev_raw": bool(df[f"is_outlier_{prefix}_dev_raw"][s:e].any()) if f"is_outlier_{prefix}_dev_raw" in df.columns else None,
            "max_vel_jump_cents": float(np.nanmax(df[f"{prefix}_vel_jump_cents"][s:e].to_numpy())) if f"{prefix}_vel_jump_cents" in df.columns else np.nan,
            "max_vel_allowed_cents": float(np.nanmax(df[f"{prefix}_vel_max_allowed_cents"][s:e].to_numpy())) if f"{prefix}_vel_max_allowed_cents" in df.columns else np.nan,
            "max_dev_cents": float(np.nanmax(df[f"{prefix}_dev_cents"][s:e].to_numpy())) if f"{prefix}_dev_cents" in df.columns else np.nan,
        })
    return pd.DataFrame(rows)

runs_stored = build_run_table(df_diag, "is_outlier_stored", prefix="old")
runs_old = build_run_table(df_diag, "is_outlier_old_recomputed", prefix="old")
runs_new = build_run_table(df_diag, "is_outlier_new", prefix="new")

print("stored runs:", len(runs_stored))
display(runs_stored.head(20))

stored runs: 337


/tmp/ipykernel_11973/3066483948.py:35: RuntimeWarning: All-NaN slice encountered
  "max_vel_jump_cents": float(np.nanmax(df[f"{prefix}_vel_jump_cents"][s:e].to_numpy())) if f"{prefix}_vel_jump_cents" in df.columns else np.nan,
/tmp/ipykernel_11973/3066483948.py:36: RuntimeWarning: All-NaN slice encountered
  "max_vel_allowed_cents": float(np.nanmax(df[f"{prefix}_vel_max_allowed_cents"][s:e].to_numpy())) if f"{prefix}_vel_max_allowed_cents" in df.columns else np.nan,
/tmp/ipykernel_11973/3066483948.py:37: RuntimeWarning: All-NaN slice encountered
  "max_dev_cents": float(np.nanmax(df[f"{prefix}_dev_cents"][s:e].to_numpy())) if f"{prefix}_dev_cents" in df.columns else np.nan,


,run_id,start_idx,end_idx_exclusive,n_samples,t_start,t_end,t_mid,any_vel_raw,any_dev_raw,max_vel_jump_cents,max_vel_allowed_cents,max_dev_cents
0,0,1254,1257,3,12.540794,12.560796,12.550795,False,False,NaN,NaN,NaN
1,1,3290,3293,3,32.902084,32.922085,32.912086,False,False,NaN,NaN,NaN
2,2,3701,3712,11,37.012344,37.112350,37.062347,False,False,NaN,NaN,NaN
3,3,4578,4581,3,45.782902,45.802902,45.792900,False,False,NaN,NaN,NaN
4,4,4660,4663,3,46.602951,46.622952,46.612953,False,False,NaN,NaN,NaN
5,5,5375,5378,3,53.753407,53.773407,53.763405,False,False,NaN,NaN,NaN
6,6,7323,7326,3,73.234642,73.254639,73.244644,False,False,NaN,NaN,NaN
7,7,7494,7498,4,74.944748,74.974747,74.959747,False,False,NaN,NaN,NaN
8,8,7680,7683,3,76.804863,76.824867,76.814865,False,False,NaN,NaN,NaN
9,9,8269,8272,3,82.695236,82.715240,82.705238,False,False,NaN,NaN,NaN


In [8]:

# files del dataframe que corresponen a un run concret
RUN_ID = 0
WINDOW_PAD_SEC = 0.15

if len(runs_stored) > 0:
    row = runs_stored.iloc[RUN_ID]
    t0 = max(float(df_diag["time_rel_sec"].min()), row["t_start"] - WINDOW_PAD_SEC)
    t1 = min(float(df_diag["time_rel_sec"].max()), row["t_end"] + WINDOW_PAD_SEC)

    cols = [
        "time_rel_sec",
        "f0_Hz",
        "f0_interpolated",
        "f0_interpolated_cents",
        "is_outlier_stored",
        "is_outlier_old_recomputed",
        "is_outlier_old_vel_raw",
        "is_outlier_old_dev_raw",
        "old_vel_jump_cents",
        "old_vel_max_allowed_cents",
        "old_dev_cents",
    ]
    cols = [c for c in cols if c in df_diag.columns]

    display(
        df_diag
        .filter((pl.col("time_rel_sec") >= t0) & (pl.col("time_rel_sec") <= t1))
        .select(cols)
        .to_pandas()
    )

,time_rel_sec,f0_Hz,f0_interpolated,f0_interpolated_cents,is_outlier_stored,is_outlier_old_recomputed,is_outlier_old_vel_raw,is_outlier_old_dev_raw,old_vel_jump_cents,old_vel_max_allowed_cents,old_dev_cents
0,12.400785,165.0,165.0,301.971434,False,False,False,False,64.127111,200.004578,142.073641
1,12.410787,175.0,175.0,403.838111,False,False,False,False,101.866677,200.023651,243.940318
2,12.420787,183.0,183.0,481.224582,False,False,False,False,77.386472,200.004578,321.326790
3,12.430787,192.0,192.0,564.339777,False,False,False,False,83.115195,200.004578,404.441985
4,12.440788,199.0,199.0,626.334321,False,False,False,False,61.994544,200.023651,466.436529
5,12.450788,203.0,203.0,660.787877,False,False,False,False,34.453556,200.004578,500.890084
6,12.460790,208.0,208.0,702.912438,False,False,False,False,42.124561,200.023651,543.014646
7,12.470790,210.0,210.0,719.479398,False,False,False,False,16.566959,200.004578,559.581605
8,12.480790,213.0,213.0,744.036321,False,False,False,False,24.556923,200.004578,584.138528
9,12.490791,213.0,213.0,744.036321,False,False,False,False,0.000000,200.023651,584.138528


## 3. Viewer comparatiu: outliers antics vs tuned

A dalt es pinta `is_outlier_stored`.

A baix es pinta `is_outlier_new`, calculat amb els controls de paràmetres.

També es mostren:
- `is_outlier_old_vel_raw` o `is_outlier_new_vel_raw`
- `is_outlier_old_dev_raw` o `is_outlier_new_dev_raw`
si els actives.

El reproductor reprodueix exactament la finestra visible.

In [9]:

def extract_audio_window(audio_path, t_start, window_sec):
    if audio_path is None:
        return None, None, t_start, t_start + window_sec, None

    audio_path = Path(audio_path)
    y, sr = sf.read(audio_path)
    audio_dur = len(y) / sr

    t0 = max(0.0, float(t_start))
    t1 = min(float(t_start + window_sec), audio_dur)

    s0 = int(round(t0 * sr))
    s1 = int(round(t1 * sr))
    y_win = y[s0:s1]
    return y_win, sr, t0, t1, audio_dur

def get_runs_from_mask(df: pl.DataFrame, mask_col: str, time_col: str = "time_rel_sec"):
    if mask_col not in df.columns:
        return []
    t = df[time_col].to_numpy()
    mask = df[mask_col].fill_null(False).to_numpy().astype(bool)
    runs = find_true_runs(mask)

    out = []
    for s, e in runs:
        x0 = t[s]
        x1 = t[e - 1]
        left = 0.5 * (t[s - 1] + t[s]) if s > 0 else x0
        right = 0.5 * (t[e - 1] + t[e]) if e < len(t) else x1
        x_mid = float(0.5 * (left + right))
        out.append({
            "start_idx": s,
            "end_idx": e,
            "x_left": float(left),
            "x_right": float(right),
            "x_mid": x_mid,
            "n_samples": int(e - s),
        })
    return out

def make_diag_df(df_base, max_velocity_sts, dev_thresh_cents, expand_neighbors):
    diag_new = compute_outlier_diagnostics(
        df_base,
        col_f0="f0_interpolated",
        max_velocity_sts=max_velocity_sts,
        dev_thresh_cents=dev_thresh_cents,
        expand_neighbors=expand_neighbors,
    )
    diag_old_local = compute_outlier_diagnostics(
        df_base,
        col_f0="f0_interpolated",
        max_velocity_sts=OLD_MAX_VELOCITY_STS,
        dev_thresh_cents=OLD_DEV_THRESH_CENTS,
        expand_neighbors=OLD_EXPAND_NEIGHBORS,
    )

    return df_base.with_columns([
        pl.Series("is_outlier_stored", df_base["is_outlier"].fill_null(False).to_numpy().astype(bool)),
        pl.Series("is_outlier_old_vel_raw", diag_old_local["is_outlier_vel_raw"]),
        pl.Series("is_outlier_old_dev_raw", diag_old_local["is_outlier_dev_raw"]),
        pl.Series("is_outlier_new", diag_new["is_outlier"]),
        pl.Series("is_outlier_new_vel_raw", diag_new["is_outlier_vel_raw"]),
        pl.Series("is_outlier_new_dev_raw", diag_new["is_outlier_dev_raw"]),
    ])

In [10]:

def plot_compare_outliers(
    df: pl.DataFrame,
    t_start: float,
    t_end: float,
    show_counts: bool = True,
    show_old_vel: bool = True,
    show_old_dev: bool = True,
    show_new_vel: bool = True,
    show_new_dev: bool = True,
):
    df_win = (
        df.filter((pl.col("time_rel_sec") >= t_start) & (pl.col("time_rel_sec") <= t_end))
          .sort("time_rel_sec")
    )
    if df_win.height == 0:
        print("No hi ha dades en aquest rang.")
        return

    t = df_win["time_rel_sec"].to_numpy()
    hz = df_win["f0_Hz_cents"].to_numpy() if "f0_Hz_cents" in df_win.columns else None
    pchip = df_win["f0_pchip_cents"].to_numpy() if "f0_pchip_cents" in df_win.columns else None
    sg13 = df_win["f0_savgol_p3_w13_cents"].to_numpy() if "f0_savgol_p3_w13_cents" in df_win.columns else None
    sg27 = df_win["f0_savgol_p3_w27_cents"].to_numpy() if "f0_savgol_p3_w27_cents" in df_win.columns else None
    interp = df_win["f0_interpolated_cents"].to_numpy() if "f0_interpolated_cents" in df_win.columns else None

    fig, axes = plt.subplots(2, 1, figsize=(15, 9), sharex=True)

    def draw_base(ax):
        if sg27 is not None:
            ax.plot(t, sg27, color="#9467bd", linewidth=3.5, alpha=0.75, zorder=5, label="sg27")
        if sg13 is not None:
            ax.plot(t, sg13, color="#2ca02c", linewidth=3.0, alpha=0.85, zorder=10, label="sg13")
        if pchip is not None:
            ax.plot(t, pchip, color="#d62728", linewidth=1.2, alpha=1, zorder=30, label="pchip")
        if interp is not None:
            ax.plot(t, interp, color="#8c564b", linewidth=1.0, alpha=0.45, zorder=32, label="interp")
        if hz is not None:
            ax.plot(t, hz, color="#001aff", linewidth=1.2, alpha=1, zorder=40, label="f0_Hz")

    def draw_mask(ax, mask_col, color, label, alpha=0.18, linestyle="--", y_text=None):
        if mask_col not in df_win.columns:
            return
        mask = df_win[mask_col].fill_null(False).to_numpy().astype(bool)
        runs = find_true_runs(mask)

        line_labeled = False
        region_labeled = False

        for s, e in runs:
            run_len = e - s
            if run_len == 1:
                x_mid = t[s]
                ax.axvline(
                    x=t[s],
                    color=color,
                    linestyle=linestyle,
                    linewidth=1.2,
                    alpha=alpha,
                    label=f"{label} (punt)" if not line_labeled else None,
                    zorder=20,
                )
                line_labeled = True
            else:
                x0 = t[s]
                x1 = t[e - 1]
                left = 0.5 * (t[s - 1] + t[s]) if s > 0 else x0
                right = 0.5 * (t[e - 1] + t[e]) if e < len(t) else x1
                x_mid = 0.5 * (left + right)
                ax.axvspan(
                    left, right,
                    color=color,
                    alpha=alpha,
                    label=f"{label} (regió)" if not region_labeled else None,
                    zorder=15,
                )
                region_labeled = True

            if show_counts and y_text is not None:
                ax.text(
                    x_mid, y_text, str(run_len),
                    ha="center", va="center", fontsize=9, color=color,
                    bbox=dict(boxstyle="round,pad=0.18", facecolor="white", edgecolor=color, alpha=0.85),
                    zorder=50, clip_on=False,
                )

    for ax in axes:
        draw_base(ax)

    for ax, main_mask, vel_mask, dev_mask, title in [
        (axes[0], "is_outlier_stored", "is_outlier_old_vel_raw", "is_outlier_old_dev_raw", "Outliers guardats / antics"),
        (axes[1], "is_outlier_new", "is_outlier_new_vel_raw", "is_outlier_new_dev_raw", "Outliers tuned / nous"),
    ]:
        ymin, ymax = ax.get_ylim()
        yrange = ymax - ymin if np.isfinite(ymax - ymin) and ymax > ymin else 1.0
        y_top = ymax - 0.06 * yrange
        y_mid = ymax - 0.12 * yrange
        y_low = ymax - 0.18 * yrange

        draw_mask(ax, main_mask, "#0055ff", "outlier", alpha=0.18, y_text=y_mid)
        if ("old" in vel_mask and show_old_vel) or ("new" in vel_mask and show_new_vel):
            draw_mask(ax, vel_mask, "#ff7f0e", "vel", alpha=0.13, y_text=y_top)
        if ("old" in dev_mask and show_old_dev) or ("new" in dev_mask and show_new_dev):
            draw_mask(ax, dev_mask, "#2ca02c", "dev", alpha=0.10, y_text=y_low)

        ax.set_ylabel("cents")
        ax.set_title(title)
        ax.grid(True, alpha=0.25)
        ax.legend(loc="best")

    axes[-1].set_xlabel("time_rel_sec")
    plt.tight_layout()
    plt.show()

In [11]:

def interactive_outlier_comparison_viewer(df_base: pl.DataFrame, audio_path, tonic_hz: float):
    t_min = float(df_base["time_rel_sec"].min())
    t_max = float(df_base["time_rel_sec"].max())

    if audio_path is not None:
        info = sf.info(str(audio_path))
        audio_dur = float(info.frames) / float(info.samplerate)
        global_max = min(t_max, audio_dur)
    else:
        audio_dur = None
        global_max = t_max

    global_min = max(0.0, t_min)

    window_slider = widgets.FloatSlider(
        value=5.0, min=0.5, max=max(0.5, min(30.0, global_max - global_min)),
        step=0.25, description="finestra (s)", continuous_update=False,
        readout_format=".2f", style={"description_width": "initial"},
        layout=widgets.Layout(width="650px"),
    )
    start_slider = widgets.FloatSlider(
        value=global_min, min=global_min, max=max(global_min, global_max - 5.0),
        step=0.25, description="inici (s)", continuous_update=False,
        readout_format=".2f", style={"description_width": "initial"},
        layout=widgets.Layout(width="650px"),
    )

    prev_button = widgets.Button(description="← finestra anterior", layout=widgets.Layout(width="180px"))
    next_button = widgets.Button(description="següent finestra →", layout=widgets.Layout(width="180px"))

    max_vel_slider = widgets.FloatSlider(
        value=NEW_MAX_VELOCITY_STS, min=20.0, max=600.0, step=5.0,
        description="new max_velocity_sts", continuous_update=False,
        style={"description_width": "initial"}, layout=widgets.Layout(width="420px"),
    )
    dev_slider = widgets.FloatSlider(
        value=NEW_DEV_THRESH_CENTS, min=50.0, max=1200.0, step=10.0,
        description="new dev_thresh_cents", continuous_update=False,
        style={"description_width": "initial"}, layout=widgets.Layout(width="420px"),
    )
    expand_slider = widgets.IntSlider(
        value=NEW_EXPAND_NEIGHBORS, min=0, max=6, step=1,
        description="new expand_neighbors", continuous_update=False,
        style={"description_width": "initial"}, layout=widgets.Layout(width="320px"),
    )

    nav_dropdown = widgets.Dropdown(
        options=[
            ("stored outlier", "is_outlier_stored"),
            ("new outlier", "is_outlier_new"),
            ("old vel raw", "is_outlier_old_vel_raw"),
            ("old dev raw", "is_outlier_old_dev_raw"),
            ("new vel raw", "is_outlier_new_vel_raw"),
            ("new dev raw", "is_outlier_new_dev_raw"),
            ("too_long_to_interp", "too_long_to_interp"),
        ],
        value="is_outlier_stored",
        description="navegar",
        style={"description_width": "initial"},
        layout=widgets.Layout(width="280px"),
    )
    prev_run_button = widgets.Button(description="← run prev", layout=widgets.Layout(width="160px"))
    next_run_button = widgets.Button(description="run next →", layout=widgets.Layout(width="160px"))

    autoplay_checkbox = widgets.Checkbox(value=False, description="autoplay àudio", indent=False)
    overlay_counts_checkbox = widgets.Checkbox(value=True, description="mostrar n samples", indent=False)
    old_vel_checkbox = widgets.Checkbox(value=True, description="mostrar old vel", indent=False)
    old_dev_checkbox = widgets.Checkbox(value=True, description="mostrar old dev", indent=False)
    new_vel_checkbox = widgets.Checkbox(value=True, description="mostrar new vel", indent=False)
    new_dev_checkbox = widgets.Checkbox(value=True, description="mostrar new dev", indent=False)

    info_box = widgets.Output()
    plot_box = widgets.Output()
    audio_box = widgets.Output()

    state = {"df_view": None, "runs_map": {}}

    def _sync_start_bounds(*args):
        w = float(window_slider.value)
        start_slider.max = max(global_min, global_max - w)
        if start_slider.value > start_slider.max:
            start_slider.value = start_slider.max

    def _recompute_df():
        df_view = make_diag_df(
            df_base,
            max_velocity_sts=float(max_vel_slider.value),
            dev_thresh_cents=float(dev_slider.value),
            expand_neighbors=int(expand_slider.value),
        )
        state["df_view"] = df_view
        state["runs_map"] = {
            col: get_runs_from_mask(df_view, col)
            for _, col in nav_dropdown.options
            if col in df_view.columns
        }
        return df_view

    def _center_window_on_x(x_mid):
        w = float(window_slider.value)
        new_start = x_mid - 0.5 * w
        new_start = max(global_min, min(new_start, start_slider.max))
        start_slider.value = new_start

    def _jump_to_run(direction=1):
        col = nav_dropdown.value
        runs = state["runs_map"].get(col, [])
        if not runs:
            return
        current_t0 = float(start_slider.value)
        current_t1 = min(current_t0 + float(window_slider.value), global_max)
        current_center = 0.5 * (current_t0 + current_t1)
        mids = np.array([r["x_mid"] for r in runs], dtype=float)

        if direction > 0:
            idxs = np.where(mids > current_center + 1e-12)[0]
            idx = int(idxs[0]) if len(idxs) > 0 else len(runs) - 1
        else:
            idxs = np.where(mids < current_center - 1e-12)[0]
            idx = int(idxs[-1]) if len(idxs) > 0 else 0
        _center_window_on_x(runs[idx]["x_mid"])

    def _go_prev(_):
        w = float(window_slider.value)
        start_slider.value = max(global_min, float(start_slider.value) - w)

    def _go_next(_):
        w = float(window_slider.value)
        start_slider.value = min(start_slider.max, float(start_slider.value) + w)

    prev_button.on_click(_go_prev)
    next_button.on_click(_go_next)
    prev_run_button.on_click(lambda _: _jump_to_run(-1))
    next_run_button.on_click(lambda _: _jump_to_run(1))
    window_slider.observe(_sync_start_bounds, names="value")

    def _refresh(*args):
        df_view = _recompute_df()

        t0 = float(start_slider.value)
        w = float(window_slider.value)
        t1 = min(t0 + w, global_max)

        info_box.clear_output(wait=True)
        plot_box.clear_output(wait=True)
        audio_box.clear_output(wait=True)

        with info_box:
            print(f"Finestra visible: {t0:.2f}s → {t1:.2f}s")
            print(f"old params: velocity={OLD_MAX_VELOCITY_STS}, dev={OLD_DEV_THRESH_CENTS}, expand={OLD_EXPAND_NEIGHBORS}")
            print(f"new params: velocity={max_vel_slider.value}, dev={dev_slider.value}, expand={expand_slider.value}")
            print(f"stored outliers: {int(df_view['is_outlier_stored'].sum())}")
            print(f"new outliers: {int(df_view['is_outlier_new'].sum())}")
            nav_col = nav_dropdown.value
            print(f"runs per '{nav_col}': {len(state['runs_map'].get(nav_col, []))}")

        with plot_box:
            plot_compare_outliers(
                df_view,
                t_start=t0,
                t_end=t1,
                show_counts=overlay_counts_checkbox.value,
                show_old_vel=old_vel_checkbox.value,
                show_old_dev=old_dev_checkbox.value,
                show_new_vel=new_vel_checkbox.value,
                show_new_dev=new_dev_checkbox.value,
            )

        y_win, sr, a0, a1, _ = extract_audio_window(audio_path, t0, t1 - t0)
        if y_win is not None:
            with audio_box:
                print(f"Àudio reproduït: {a0:.2f}s → {a1:.2f}s")
                display(Audio(data=y_win.T if y_win.ndim == 2 else y_win, rate=sr, autoplay=autoplay_checkbox.value))
                display(Javascript("""
                    setTimeout(() => {
                        const audios = document.querySelectorAll("audio");
                        if (audios.length > 0) {
                            const a = audios[audios.length - 1];
                            a.playbackRate = 0.5;
                        }
                    }, 100);
                """))

    for w in [
        start_slider, window_slider, max_vel_slider, dev_slider, expand_slider,
        nav_dropdown, autoplay_checkbox, overlay_counts_checkbox,
        old_vel_checkbox, old_dev_checkbox, new_vel_checkbox, new_dev_checkbox
    ]:
        w.observe(_refresh, names="value")

    controls = widgets.VBox([
        window_slider,
        start_slider,
        widgets.HBox([prev_button, next_button]),
        widgets.HBox([max_vel_slider, dev_slider, expand_slider]),
        widgets.HBox([nav_dropdown, prev_run_button, next_run_button]),
        widgets.HBox([autoplay_checkbox, overlay_counts_checkbox, old_vel_checkbox, old_dev_checkbox, new_vel_checkbox, new_dev_checkbox]),
    ])

    display(controls, info_box, plot_box, audio_box)
    _sync_start_bounds()
    _refresh()

In [12]:

interactive_outlier_comparison_viewer(
    df_base=df_debug,
    audio_path=AUDIO_PATH,
    tonic_hz=TONIC_HZ,
)

Output()

Output()

Output()